# 2. Фурье-пространство и перенос решения на приёмник

Здесь ещё не строим пространственную таблицу. Цель — получить обычную
линейную систему на одном $(k,\omega)$ и понять, почему она не зависит
от направления первоначального фотона.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 2.1. Соглашение о Фурье
$$\widetilde f(\mathbf k,\omega)=\int f(\mathbf x,t)e^{-i\mathbf k\cdot\mathbf x+i\omega t}d^3xdt.$$
Тогда $\partial_t\to-i\omega$, $\nabla\to i\mathbf k$. В однородной среде
разные $\mathbf k$ не связаны:
$$(D-V)\widetilde I=\widetilde q,\quad
D=d_0+ik\mu,\quad d_0=\mu_t-i\omega/v,\quad\mu=\widehat{\mathbf k}\cdot\mathbf s.$$
$V$ — интегральный оператор рассеяния, $(Vf)(\mathbf s)=\mu_s\int p(\mathbf s\cdot\mathbf s')f(\mathbf s')d\Omega'$.

Приёмник измеряет $\widetilde K=\int\widetilde I\,d\Omega$.
С билинейным спариванием $[f,g]=\int fg\,d\Omega$ (без комплексного сопряжения)
оператор $D-V$ симметричен. Для направленной вспышки:
$$\widetilde K=[1,(D-V)^{-1}\delta_{\mathbf s_0}]
=[(D-V)^{-1}1,\delta_{\mathbf s_0}]=h(\mathbf s_0),\qquad(D-V)h=1.$$

Правая часть единица соответствует изотропному приёмнику. Выбирая ось
вдоль $\mathbf k$, получаем только $m=0$. Произвольный $\mathbf s_0$
подставляется **после** решения. Для направленного приёмника правая часть
была бы его функцией чувствительности; в общем случае появились бы другие $m$.

In [ ]:
k=.2;omega=.03
v=medium.speed_m_per_ns
d0=medium.extinction_per_m-1j*omega/v
print('k [m^-1] =',k,'d0 [m^-1] =',d0)
mu=np.linspace(-1,1,501)
free=1/(d0+1j*k*mu)
fig,ax=plt.subplots();ax.plot(mu,free.real,label='Re 1/D');ax.plot(mu,free.imag,label='Im 1/D');ax.set(xlabel='mu',ylabel='length [m]');ax.legend();plt.show()

## 2.2. Свободные моменты и независимое интегрирование

Положим $p_\ell(\mu)=\sqrt{(2\ell+1)/2}P_\ell(\mu)$,
$\int p_\ell p_jd\mu=\delta_{\ell j}$. Не путать $p_\ell$ с фазовой функцией $p(x)$.
$$b_\ell=\int_{-1}^1\frac{p_\ell(\mu)}{D(\mu)}d\mu.$$
$b_\ell$ — коэффициенты свободного **детекторного** отклика, не правые
части уравнения для направленного источника. Сверим рекурсию с прямой квадратурой.

In [ ]:
from scipy.special import eval_legendre, roots_legendre
from lighthit.angular import free_moments_and_tail
ell=np.arange(7);x,w=roots_legendre(512)
p=eval_legendre(ell[:,None],x)*np.sqrt((2*ell+1)/2)[:,None]
b_quad=p@(w/(d0+1j*k*x))
b,tail=free_moments_and_tail([k],d0,6)
print(np.column_stack([ell,b[0],b_quad]))
np.testing.assert_allclose(b[0],b_quad,rtol=1e-11,atol=1e-12)

## 2.3. Почему оператор рассеяния диагонален

Это следует из зависимости ядра только от скалярного произведения направлений.
Для любой нормированной неотрицательной фазовой функции
$$\chi_\ell=2\pi\int_{-1}^1p(x)P_\ell(x)dx,\qquad \gamma_\ell=\mu_s\chi_\ell.$$
В HG-модели $\chi_\ell=g^\ell$. В частности, $\gamma_0=\mu_s$.
Ниже эта формула проверяется квадратурой, а не предполагается при построении таблицы.

In [ ]:
from lighthit.single import hg_phase
chi=2*np.pi*(eval_legendre(ell[:,None],x)*(w*hg_phase(x,medium.g))[None,:]).sum(1)
print(np.column_stack([ell,chi,medium.g**ell]))
np.testing.assert_allclose(chi,medium.g**ell,rtol=1e-11)

## Задания

Доказать перенос на приёмник ещё раз, используя эрмитово скалярное
произведение. Показать, как исчезает итоговое сопряжение при переходе от
$\psi$ к $h=\psi^*$. Вычислить $\chi_\ell$ для другой положительной
фазовой функции. Объяснить, почему диагональность не является привилегией HG.

Код: `angular.py`; книга: разделы Fourier / Reciprocity главы 2.